# Sesión 5 EXTRA - Visualizando algoritmos DQN en Acción en Google Colab (directamente en notebooks)

> En la presente sesión, se va a desarrollar la **implementación** de los algoritmos básicos de aprendizaje por refuerzo: **Deep Q-networks (DQN)**, y **Policy Gradients (PG)**. El caso de uso desarrollado consistirá en el **Atary**, cuyo **entorno** es modelado por la librería de acceso público de **OpenAI gym**. Se hará uso de dos librería para la implementación de algoritmos de aprendizaje por refuerzo: **keras-rl** y **pytorch**. La primera, de más alto nivel, servirá para establecer un conocimiento práctico de los métodos DQN, y analizar los distintos hiperparámetros de entrenamiento. En el segundo caso, PG será implemetado con pytorch, librería de más bajo nivel, y la cuál permitirá trabajar en detalle la estrategia de aprendizaje on-policy y la optimización por gradiente de la policy.






---
## **PARTE 1** - Instalación y requisitos previos

> Las prácticas han sido preparadas para poder realizarse en el entorno de trabajo de Google Colab. Sin embargo, esta plataforma presenta ciertas incompatibilidades a la hora de visualizar la renderización en gym. Por ello, para obtener estas visualizaciones, se deberá trasladar el entorno de trabajo a local. Por ello, el presente dosier presenta instrucciones para poder trabajar en ambos entornos. Siga los siguientes pasos para un correcto funcionamiento:
1.   **LOCAL:** Preparar el enviroment, siguiendo las intrucciones detalladas en la sección *1.1.Preparar enviroment*.
2.  **AMBOS:** Modificar las variables "mount" y "drive_mount" a la carpeta de trabajo en drive en el caso de estar en Colab, y ejecturar la celda *1.2.Localizar entorno de trabajo*.
3. **COLAB:** se deberá ejecutar las celdas correspondientes al montaje de la carpeta de trabajo en Drive. Esta corresponde a la sección *1.3.Montar carpeta de datos local*.
4.  **AMBOS:** Instalar las librerías necesarias, siguiendo la sección *1.4.Instalar librerías necesarias*.



---
### 1.1. Preparar enviroment (solo local)



> Para preparar el entorno de trabajo en local, se han seguido los siguientes pasos:
1. En Windows, puede ser necesario instalar las C++ Build Tools. Para ello, siga los siguientes pasos: https://towardsdatascience.com/how-to-install-openai-gym-in-a-windows-environment-338969e24d30.
2. Instalar Anaconda
3. Siguiendo el código que se presenta comentado en la próxima celda: Crear un enviroment, cambiar la ruta de trabajo, e instalar librerías básicas.


```
conda update --all
conda create --name miar_rl python=3.8
conda activate miar_rl
cd "PATH_TO_FOLDER"
conda install git
pip install jupyter
```


4. Abrir la notebook con *jupyter-notebook*.



```
jupyter-notebook
```




---
### 1.2. Localizar entorno de trabajo: Google colab o local

In [ ]:
# ATENCIÓN!! Modificar ruta relativa a la práctica si es distinta (drive_root)
mount='/content/gdrive'
drive_root = mount + "/My Drive/VIU/08_AR_MIAR/sesiones_practicas/sesion_practica_1"

try:
  from google.colab import drive
  IN_COLAB=True
except:
  IN_COLAB=False

---
### 1.3. Montar carpeta de datos local (solo Colab)

In [ ]:
# Switch to the directory on the Google Drive that you want to use
import os
if IN_COLAB:
  print("We're running Colab")

  if IN_COLAB:
    # Mount the Google Drive at mount
    print("Colab: mounting Google drive on ", mount)

    drive.mount(mount)

    # Create drive_root if it doesn't exist
    create_drive_root = True
    if create_drive_root:
      print("\nColab: making sure ", drive_root, " exists.")
      os.makedirs(drive_root, exist_ok=True)

    # Change to the directory
    print("\nColab: Changing directory to ", drive_root)
    %cd $drive_root
# Verify we're in the correct working directory
%pwd
print("Archivos en el directorio: ")
print(os.listdir())

We're running Colab
Colab: mounting Google drive on  /content/gdrive
Mounted at /content/gdrive

Colab: making sure  /content/gdrive/My Drive/VIU/08_AR_MIAR/sesiones_practicas/sesion_practica_1  exists.

Colab: Changing directory to  /content/gdrive/My Drive/VIU/08_AR_MIAR/sesiones_practicas/sesion_practica_1
/content/gdrive/My Drive/VIU/08_AR_MIAR/sesiones_practicas/sesion_practica_1
Archivos en el directorio: 
['dqn_BreakoutDeterministic-v4_log.json', 'dqn_BreakoutDeterministic-v4_weights.h5f.index', 'dqn_BreakoutDeterministic-v4_weights.h5f.data-00000-of-00001', 'dqn_CartPole-v1_weights.h5f.index', 'dqn_CartPole-v1_weights.h5f.data-00000-of-00001', 'checkpoint']


---
### 1.4. Instalar librerías necesarias


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
if IN_COLAB:
  %pip install gym==0.17.3
  %pip install git+https://github.com/Kojoley/atari-py.git
  %pip install keras-rl2==1.0.5
  %pip install tensorflow==2.8
else:
  %pip install gym==0.17.3
  %pip install git+https://github.com/Kojoley/atari-py.git
  %pip install pyglet==1.5.0
  %pip install h5py==3.1.0
  %pip install Pillow==9.5.0
  %pip install keras-rl2==1.0.5
  %pip install Keras==2.2.4
  %pip install tensorflow==2.5.3
  %pip install torch==2.0.1
  %pip install agents==1.4.0

  Cloning https://github.com/Kojoley/atari-py.git to /tmp/pip-req-build-k0q9_4jv
  Running command git clone --filter=blob:none --quiet https://github.com/Kojoley/atari-py.git /tmp/pip-req-build-k0q9_4jv
  Resolved https://github.com/Kojoley/atari-py.git to commit 86a1e05c0a95e9e6233c3a413521fdb34ca8a089
  Preparing metadata (setup.py) ... done
  Created wheel for atari-py: filename=atari_py-1.2.2-cp310-cp310-linux_x86_64.whl size=4738553 sha256=7a7643e92679306a9153154bef1f720ea27ebcd960bf4e43d0e2e9d68fdac247
  Stored in directory: /tmp/pip-ephem-wheel-cache-e7dgfg2d/wheels/6c/9b/f1/8a80a63a233d6f4eb2e2ee8c7357f4875f21c3ffc7f2613f9f
Successfully built atari-py
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.6/497.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 18.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.5/462.5 kB 24.0 MB/s eta 0:00:00
     ━━

---
### 1.5.Acerca de las librerías para RL

Librería para trabajar con nuestros entornos: gym (https://gym.openai.com/) \
Librería para trabajar con deep learning: tensorflow (https://www.tensorflow.org/) \
Librería para desarrollar soluciones de RL a alto nivel: keras-rl (https://github.com/keras-rl/keras-rl) \


---
## **PARTE 2** - *Deep Q-learning*


---
### 2.1. DQN Pseudo-código

In [ ]:
from IPython import display
display.Image("images/dqn.png", width = 600, height = 300)

---
### 2.3. Ejemplo Carga Agente DQN Pre-Entrenado con Keras-rl - Breakout

Información del entorno: https://www.gymlibrary.dev/environments/atari/breakout/

Basado en: https://github.com/Finspire13/pytorch-policy-gradient-example/blob/master/pg.py

In [ ]:
from __future__ import division

from PIL import Image
import numpy as np
import gym

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, Convolution2D, Permute
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import FileLogger, ModelIntervalCheckpoint

In [ ]:
INPUT_SHAPE = (84, 84)
WINDOW_LENGTH = 4

In [ ]:
# In this example, we need to preprocess the observations
class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3  # (height, width, channel)
        img = Image.fromarray(observation)
        img = img.resize(INPUT_SHAPE).convert('L')
        processed_observation = np.array(img)
        assert processed_observation.shape == INPUT_SHAPE
        return processed_observation.astype('uint8')

    def process_state_batch(self, batch):
        processed_batch = batch.astype('float32') / 255.
        return processed_batch

    def process_reward(self, reward):
        return np.clip(reward, -1., 1.)

In [ ]:
# Get the environment and extract the number of actions.
import gym
import numpy as np
env_name = 'BreakoutDeterministic-v4'
env = gym.make(env_name)
np.random.seed(123)
env.seed(123)
nb_actions = env.action_space.n

In [ ]:
print("Numero de acciones disponibles: " + str(nb_actions))

Numero de acciones disponibles: 4


In [ ]:
print("Formato de las observaciones:")
env.observation_space

Formato de las observaciones:


Box(0, 255, (210, 160, 3), uint8)

In [ ]:
# Next, we build our model. We use the same model that was described by Mnih et al. (2015).
input_shape = (WINDOW_LENGTH,) + INPUT_SHAPE
model = Sequential()
print(K.image_data_format())
if K.image_data_format() == 'channels_last':
    # (width, height, channels)
    model.add(Permute((2, 3, 1), input_shape=input_shape))
elif K.image_data_format() == 'channels_first':
    # (channels, width, height)
    model.add(Permute((1, 2, 3), input_shape=input_shape))
else:
    raise RuntimeError('Unknown image_dim_ordering.')

model.add(Convolution2D(32, (8, 8), strides=(4, 4)))
model.add(Activation('relu'))
model.add(Convolution2D(64, (4, 4), strides=(2, 2)))
model.add(Activation('relu'))
model.add(Convolution2D(64, (3, 3), strides=(1, 1)))
model.add(Activation('relu'))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dense(nb_actions))
model.add(Activation('linear'))
print(model.summary())

channels_last
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 permute (Permute)           (None, 84, 84, 4)         0         
                                                                 
 conv2d (Conv2D)             (None, 20, 20, 32)        8224      
                                                                 
 activation (Activation)     (None, 20, 20, 32)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 9, 64)          32832     
                                                                 
 activation_1 (Activation)   (None, 9, 9, 64)          0         
                                                                 
 conv2d_2 (Conv2D)           (None, 7, 7, 64)          36928     
                                                                 
 activation_2 (Activation)   (None, 7, 7, 

In [ ]:
memory = SequentialMemory(limit=1000000, window_length=WINDOW_LENGTH)
processor = AtariProcessor()

In [ ]:
policy = LinearAnnealedPolicy(EpsGreedyQPolicy(), attr='eps',
                              value_max=1., value_min=.1, value_test=.05,
                              nb_steps=1000000)

In [ ]:
dqn = DQNAgent(model=model, nb_actions=nb_actions, policy=policy,
               memory=memory, processor=processor,
               nb_steps_warmup=50000, gamma=.99,
               target_model_update=10000,
               train_interval=4)
dqn.compile(Adam(learning_rate=.00025), metrics=['mae'])

In [ ]:
# Training part - We skip training and go directly to visualized pre-trained Agent in Action
#weights_filename = 'dqn_{}_weights.h5f'.format(env_name)
#checkpoint_weights_filename = 'dqn_' + env_name + '_weights_{step}.h5f'
#log_filename = 'dqn_{}_log.json'.format(env_name)
#callbacks = [ModelIntervalCheckpoint(checkpoint_weights_filename, interval=250000)]
#callbacks += [FileLogger(log_filename, interval=100)]
#
#dqn.fit(env, callbacks=callbacks, nb_steps=1750000, log_interval=10000, visualize=False)
#
#dqn.save_weights(weights_filename, overwrite=True)

In [ ]:
# Testing part - Point to the right Weights File
#weights_filename = 'dqn_{}_weights.h5f'.format(env_name)
weights_filename = "/content/drive/My Drive/08_MIAR/sesiones_practicas/sesion_practica_1/dqn_BreakoutDeterministic-v4_weights_1750000.h5f"
dqn.load_weights(weights_filename)
#dqn.test(env, nb_episodes=10, visualize=False)



---



---
### 2.4 Visualize in Action

[More details on SO](https://stackoverflow.com/questions/52726475/display-openai-gym-in-jupyter-notebook-only)

#### 2.4.1 As Cell GIF animation

In [ ]:
import cv2
import IPython.display as display
import ipywidgets as widgets
import time


def render_game_to_cell(obs):
    screen_data = obs
    frame = cv2.cvtColor(screen_data, cv2.COLOR_RGB2BGR)
    _, jpeg = cv2.imencode('.jpeg', frame)
    image_widget.value = jpeg.tobytes()

In [ ]:
image_widget = widgets.Image(format='jpeg')
display.display(image_widget)

Image(value=b'', format='jpeg')

In [ ]:
env.reset()
action = env.action_space.sample()  # your agent here (this takes first actions as random action)
obs, rew, done, info = env.step(action)

while not done:
    obs, rew, done, info = env.step(dqn.forward(processor.process_observation(obs))) # get actions as a forward pass of your model
    render_game_to_cell(obs)
    time.sleep(0.05)

#### 2.4.2 Record agent in a MP4 Video and play it after episode finishes

Alternatively use this snipped to export your play as a MP4 video and share it. Point to the right output folder in your setup.

In [ ]:
import gym
from gym import wrappers

env = gym.make('SpaceInvaders-v0')
env = wrappers.Monitor(env, "/content/drive/My Drive/08_MIAR/sesiones_practicas/sesion_practica_1/", force=True)
env.reset()
for _ in range(1000):
    action = env.action_space.sample()
    observation, reward, done, info = env.step(action)
    if done: break
env.close()

/usr/lib/python3.10/subprocess.py:1796: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _posixsubprocess.fork_exec(


In [ ]:
#Alternatively, play your video first in a cell with this code.
#
##############
import io
import base64
from IPython.display import HTML

video = io.open('/content/drive/My Drive/08_MIAR/sesiones_practicas/sesion_practica_1/openaigym.video.%s.video000000.mp4' % env.file_infix, 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''
    <video width="360" height="auto" alt="test" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>'''
.format(encoded.decode('ascii')))

---